In [ ]:
import os
os.chdir("../")

In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse import load_npz
import mlflow
import mlflow.sklearn
from sklearn.neighbors import NearestNeighbors

In [ ]:
# load train matrix
train_matrix = load_npz("../data/processed/train_matrix.npz")

# load mappings
user_map = pd.read_csv("../data/processed/mappings/user_map.csv")
item_map = pd.read_csv("../data/processed/mappings/item_map.csv")

# load test data
test_data = pd.read_csv("../data/processed/test_data.csv")

print("Train matrix shape:", train_matrix.shape)
print("Test data shape:", test_data.shape)

In [ ]:
if not os.path.exists("mlflow"):
    os.makedirs("mlflow")

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("movie-recommendation")

print("MLflow tracking URI set!")

In [ ]:
from sklearn.metrics import mean_squared_error

# convert item_map to dictionary for fast lookup
item_to_idx = dict(zip(item_map["movie_id"], item_map["item_idx"]))
idx_to_item = dict(zip(item_map["item_idx"], item_map["movie_id"]))
user_to_idx = dict(zip(user_map["user_id"], user_map["user_idx"]))

# try different K values
k_values = [10, 20, 50]

for k in k_values:
    with mlflow.start_run(run_name=f"ItemKNN_K{k}"):
        
        # train model
        model = NearestNeighbors(n_neighbors=k, metric="cosine", algorithm="brute")
        model.fit(train_matrix.T)
        
        # log parameter
        mlflow.log_param("k", k)
        mlflow.log_param("metric", "cosine")
        
        print(f"K={k} trained and logged to MLflow.")

In [ ]:
from sklearn.metrics import root_mean_squared_error

for k in k_values:
    with mlflow.start_run(run_name=f"ItemKNN_K{k}_eval"):
        
        # train model
        model = NearestNeighbors(n_neighbors=k, metric="cosine", algorithm="brute")
        model.fit(train_matrix.T)
        
        # log parameters
        mlflow.log_param("k", k)
        mlflow.log_param("metric", "cosine")
        
        # evaluate on test data
        actuals = []
        predictions = []
        
        for _, row in test_data.head(1000).iterrows():
            user_idx = user_to_idx.get(row["user_id"])
            item_idx = item_to_idx.get(row["movie_id"])
            
            if user_idx is None or item_idx is None:
                continue
                
            # get similar items
            distances, indices = model.kneighbors(
                train_matrix.T[item_idx], 
                n_neighbors=k
            )
            
            # predict as weighted average
            similar_ratings = train_matrix[user_idx, indices[0]].toarray().flatten()
            weights = 1 - distances[0]
            
            if weights.sum() > 0:
                pred = np.average(similar_ratings, weights=weights)
            else:
                pred = train_matrix[user_idx].mean()
                
            actuals.append(row["rating"])
            predictions.append(pred)
        
        # calculate RMSE
        rmse = root_mean_squared_error(actuals, predictions)
        mlflow.log_metric("rmse", rmse)
        
        print(f"K={k} → RMSE: {rmse:.4f}")

### SVD

In [ ]:
from scipy.sparse.linalg import svds

with mlflow.start_run(run_name="SVD"):
    
    # number of latent factors
    n_factors = 50
    
    # train SVD
    U, sigma, Vt = svds(train_matrix.astype(float), k=n_factors)
    
    # convert sigma to diagonal matrix
    sigma = np.diag(sigma)
    
    # reconstruct full predictions matrix
    predicted_ratings = np.dot(np.dot(U, sigma), Vt)
    
    # log parameter
    mlflow.log_param("n_factors", n_factors)
    mlflow.log_param("model", "SVD")
    
    print(f"SVD trained and logged to MLflow.")
    print(f"Predicted ratings matrix shape: {predicted_ratings.shape}")

In [ ]:

with mlflow.start_run(run_name="SVD_eval"):
    
    mlflow.log_param("n_factors", 50)
    mlflow.log_param("model", "SVD")
    
    actuals = []
    predictions = []
    
    for _, row in test_data.head(1000).iterrows():
        user_idx = user_to_idx.get(row["user_id"])
        item_idx = item_to_idx.get(row["movie_id"])
        
        if user_idx is None or item_idx is None:
            continue
        
        pred = predicted_ratings[user_idx, item_idx]
        actuals.append(row["rating"])
        predictions.append(pred)
    
    rmse = root_mean_squared_error(actuals, predictions)
    mlflow.log_metric("rmse", rmse)
    
    print(f"SVD → RMSE: {rmse:.4f}")

In [ ]:
import joblib
import os

# train final model with best K
best_model = NearestNeighbors(n_neighbors=50, metric="cosine", algorithm="brute")
best_model.fit(train_matrix.T)

# save model
os.makedirs("models", exist_ok=True)
joblib.dump(best_model, "models/itemknn_k50.joblib")

# save predicted ratings matrix
np.save("models/train_matrix_dense.npy", train_matrix.toarray())

print("Best model saved!")

### Predicted ratings matrix

In [ ]:
movies = pd.read_csv("data/raw/ml-1m/movies.dat", sep='::', engine='python', header=None, names=['movie_id', 'title', 'genres'], encoding='latin-1')

In [ ]:
def recommend_movies(user_id, n_recommendations=10):
    
    # get user index
    user_idx = user_to_idx.get(user_id)
    if user_idx is None:
        print(f"User {user_id} not found!")
        return
    
    # get user ratings from train matrix
    user_ratings = train_matrix[user_idx].toarray().flatten()
    
    # find movies user already watched
    watched_movies = np.where(user_ratings > 0)[0]
    
    # find movies user has NOT watched
    unwatched_movies = np.where(user_ratings == 0)[0]
    
    # predict ratings for unwatched movies
    predicted_scores = []
    
    for item_idx in unwatched_movies:
        # get similar movies to this unwatched movie
        distances, indices = best_model.kneighbors(
            train_matrix.T[item_idx],
            n_neighbors=50
        )
        
        # get user ratings for similar movies
        similar_ratings = user_ratings[indices[0]]
        weights = 1 - distances[0]
        
        if weights.sum() > 0 and similar_ratings.sum() > 0:
            pred = np.average(similar_ratings, weights=weights)
        else:
            pred = 0
            
        predicted_scores.append((item_idx, pred))
    
    # sort by predicted score
    predicted_scores.sort(key=lambda x: x[1], reverse=True)
    
    # get top N recommendations
    top_n = predicted_scores[:n_recommendations]
    
    # convert back to movie titles
    print(f"\nTop {n_recommendations} recommendations for User {user_id}:\n")
    for item_idx, score in top_n:
        movie_id = idx_to_item.get(item_idx)
        title = movies[movies["movie_id"] == movie_id]["title"].values
        if len(title) > 0:
            print(f"→ {title[0]}  (predicted score: {score:.2f})")


In [15]:
# test with user 1
recommend_movies(1)


Top 10 recommendations for User 1:

→ Lion King, The (1994)  (predicted score: 1.67)
→ Jungle Book, The (1967)  (predicted score: 1.36)
→ Pinocchio (1940)  (predicted score: 1.36)
→ Mary Poppins (1964)  (predicted score: 1.31)
→ Prince of Egypt, The (1998)  (predicted score: 1.30)
→ Sleeping Beauty (1959)  (predicted score: 1.29)
→ Fantasia 2000 (1999)  (predicted score: 1.28)
→ Lady and the Tramp (1955)  (predicted score: 1.27)
→ Anastasia (1997)  (predicted score: 1.26)
→ 101 Dalmatians (1961)  (predicted score: 1.25)


In [16]:
# see what user 1 already watched
user_idx = user_to_idx.get(1)
user_ratings = train_matrix[user_idx].toarray().flatten()
watched = np.where(user_ratings > 0)[0]

print(f"User 1 watched {len(watched)} movies:\n")
for item_idx in watched[:10]:
    movie_id = idx_to_item.get(item_idx)
    title = movies[movies["movie_id"] == movie_id]["title"].values
    rating = user_ratings[item_idx]
    if len(title) > 0:
        print(f"→ {title[0]}  (rated: {rating})")

User 1 watched 42 movies:

→ Airplane! (1980)  (rated: 4)
→ Princess Bride, The (1987)  (rated: 3)
→ Mulan (1998)  (rated: 4)
→ E.T. the Extra-Terrestrial (1982)  (rated: 4)
→ Ferris Bueller's Day Off (1986)  (rated: 4)
→ Wizard of Oz, The (1939)  (rated: 4)
→ Toy Story (1995)  (rated: 5)
→ Bug's Life, A (1998)  (rated: 5)
→ Back to the Future (1985)  (rated: 5)
→ Erin Brockovich (2000)  (rated: 4)


In [17]:
recommend_movies(100)


Top 10 recommendations for User 100:

→ Braveheart (1995)  (predicted score: 1.52)
→ Star Trek: The Wrath of Khan (1982)  (predicted score: 1.39)
→ Glory (1989)  (predicted score: 1.38)
→ Star Trek IV: The Voyage Home (1986)  (predicted score: 1.34)
→ Dances with Wolves (1990)  (predicted score: 1.33)
→ Fugitive, The (1993)  (predicted score: 1.29)
→ Thelma & Louise (1991)  (predicted score: 1.28)
→ Die Hard (1988)  (predicted score: 1.28)
→ Hunt for Red October, The (1990)  (predicted score: 1.27)
→ L.A. Confidential (1997)  (predicted score: 1.26)


In [18]:
# see what user 1 already watched
user_idx = user_to_idx.get(100)
user_ratings = train_matrix[user_idx].toarray().flatten()
watched = np.where(user_ratings > 0)[0]

print(f"User 100 watched {len(watched)} movies:\n")
for item_idx in watched[:10]:
    movie_id = idx_to_item.get(item_idx)
    title = movies[movies["movie_id"] == movie_id]["title"].values
    rating = user_ratings[item_idx]
    if len(title) > 0:
        print(f"→ {title[0]}  (rated: {rating})")


User 100 watched 61 movies:

→ Shawshank Redemption, The (1994)  (rated: 4)
→ Star Wars: Episode VI - Return of the Jedi (1983)  (rated: 4)
→ Princess Bride, The (1987)  (rated: 4)
→ Star Wars: Episode V - The Empire Strikes Back (1980)  (rated: 4)
→ Meet the Parents (2000)  (rated: 3)
→ Fargo (1996)  (rated: 3)
→ Conspiracy Theory (1997)  (rated: 2)
→ GoodFellas (1990)  (rated: 4)
→ Men in Black (1997)  (rated: 2)
→ Indiana Jones and the Temple of Doom (1984)  (rated: 3)


* Model is working correctly!
* User 1 likes animated/family movies
* Model correctly recommends similar animated movies
* Predicted scores are low due to no rating normalization
* But ranking order is correct which is what matters